# 🐺 Bernard Decision Model — Fine-Tune Qwen3.5-2B with Unsloth

Fine-tune Qwen3.5-2B on Bernard's autonomous decision data (RALPH traces, Night Worker, audit trail).

**Requirements:** Google Colab with T4 GPU (free tier works)

**Output:** GGUF model for Ollama deployment on Mac

In [ ]:
# ── 1. Install Unsloth ──
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo

In [ ]:
# ── 2. Load Qwen3.5-2B ──
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3.5-2B",
    max_seq_length=2048,
    load_in_4bit=True,  # QLoRA — fits in T4 16GB
    dtype=None,  # auto-detect
)

print(f"✅ Model loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ── 3. Attach LoRA Adapter ──
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    use_rslora=False,
)

print(f"✅ LoRA attached. Trainable params: {model.print_trainable_parameters()}")

In [ ]:
# ── 4. Download & Load Dataset ──
import json

# Download from GitHub
!wget -q https://raw.githubusercontent.com/AlekTkT/bernard-finetune/main/bernard-decision-train.jsonl
!wget -q https://raw.githubusercontent.com/AlekTkT/bernard-finetune/main/bernard-decision-eval.jsonl

# Parse JSONL
def load_jsonl(filename):
    examples = []
    with open(filename, 'r') as f:
        for line in f:
            if line.strip():
                examples.append(json.loads(line))
    return examples

train_data = load_jsonl('bernard-decision-train.jsonl')
eval_data = load_jsonl('bernard-decision-eval.jsonl')

print(f"✅ Train: {len(train_data)} examples, Eval: {len(eval_data)} examples")

In [ ]:
# ── 5. Format Dataset for Training ──
from unsloth.chat_templates import get_chat_template, standardize_sharegpt
from datasets import Dataset

tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
)

def format_example(example):
    """Apply chat template to messages."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset = Dataset.from_list(train_data).map(format_example)
eval_dataset = Dataset.from_list(eval_data).map(format_example)

print(f"✅ Formatted. Sample:")
print(train_dataset[0]["text"][:500])

In [ ]:
# ── 6. Training ──
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        learning_rate=2e-4,
        optim="adamw_8bit",
        weight_decay=0.01,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
    ),
)

print("🚀 Starting training...")
stats = trainer.train()
print(f"\n✅ Training complete!")
print(f"   Loss: {stats.training_loss:.4f}")
print(f"   Runtime: {stats.metrics['train_runtime']:.0f}s")
print(f"   Samples/sec: {stats.metrics['train_samples_per_second']:.1f}")

In [ ]:
# ── 7. Test Before Export ──
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": "Tu es Bernard, un agent IA autonome spécialisé en trading crypto, DevOps et automatisation."},
    {"role": "user", "content": "Contexte RALPH:\nSignal score: 8/10\nSource: errors\nSignaux actifs: 7\nActions aujourd'hui: 3\nCatégorie: health\nFenêtre: nuit (travail lourd autorisé)\n\nQuelle action exécuter maintenant ?"},
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("🧪 Test inference:")
print(response)

In [ ]:
# ── 8. Export GGUF for Ollama ──
print("📦 Exporting GGUF q4_k_m (optimized for Mac)...")

model.save_pretrained_gguf(
    "bernard-decision-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

print("✅ GGUF exported!")

# Also save LoRA adapter for future merging
model.save_pretrained("bernard-decision-lora")
tokenizer.save_pretrained("bernard-decision-lora")
print("✅ LoRA adapter saved!")

In [ ]:
# ── 9. Download GGUF ──
import glob

gguf_files = glob.glob("bernard-decision-gguf/*.gguf")
print(f"📥 GGUF files: {gguf_files}")

for f in gguf_files:
    files.download(f)
    print(f"   Downloaded: {f}")

print("\n🎉 Done! Next steps:")
print("1. Place the .gguf file in ~/openclaw/fine-tuning/models/")
print("2. Run: ollama create bernard-decision -f Modelfile")
print("3. Test: ollama run bernard-decision")